# 03. 集成学习体系

集成学习的思想很直接：单个模型不稳定，那就把多个模型组合起来。

## 先建立直觉

            集成学习可以理解成“不要只听一个专家的意见”。

单个模型可能会：

- 看走眼
- 对训练数据过于敏感
- 在某些局部模式上犯系统性错误

如果把多个模型组合起来，就有机会让它们互相纠错。

所以集成学习最核心的思想是：

- 要么让多个模型从不同角度看问题，然后投票
- 要么让后面的模型专门修正前面的错误

## 架构是怎么工作的

            集成学习主要有两条路线：

- Bagging：多个模型并行训练，再做平均或投票
- Boosting：多个模型串行训练，后一个接着修前一个

随机森林属于 Bagging 路线，本质上是“很多棵不一样的树一起投票”。
AdaBoost 和 GBDT 属于 Boosting 路线，本质上是“逐步叠加弱学习器，让整体越来越强”。

## 训练时到底发生了什么

            Bagging 训练时，会给每个子模型喂不同抽样的数据，目的是制造差异性。
只有子模型之间有差异，投票才有意义。

Boosting 则更像一条纠错流水线：

- 第一个模型先学
- 第二个模型重点关注第一个模型没学好的部分
- 第三个模型继续修正前面模型的残差或错误

所以 Bagging 更偏向“降方差”，Boosting 更偏向“降偏差”。

## 什么时候该用它

            如果你做的是表格数据任务，集成学习往往是极强的基线。

- 随机森林：稳、好调、鲁棒
- AdaBoost：结构简单时有优势
- GBDT：通常精度更高，但调参更敏感

在很多工业表格场景里，集成模型的表现非常强。

## 最常见的误区

            常见误区：

1. **误以为模型越多越好**
   如果弱学习器几乎一样，增加数量收益会递减。

2. **误以为 Boosting 一定优于 Bagging**
   有噪声、数据量不大、场景不稳定时，Boosting 也可能更容易过拟合。

3. **误以为集成模型就不需要解释**
   虽然比单棵树复杂，但仍然可以通过特征重要性、SHAP 等方法做解释。

## 1. 集成学习的核心直觉

- Bagging：并行训练多个模型，再做投票或平均，主要用于**降方差**
- Boosting：串行训练多个模型，后一个模型重点纠正前一个模型的错误

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_moon, y_moon = make_moons(n_samples=500, noise=0.28, random_state=42)

moon_models = {
    "单棵决策树": DecisionTreeClassifier(max_depth=3, random_state=42),
    "随机森林": RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=150,
        learning_rate=0.8,
        random_state=42,
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

for model in moon_models.values():
    model.fit(X_moon, y_moon)

In [ ]:
def plot_cls_boundary(ax, model, X, y, title):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    pred = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, pred, alpha=0.30, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=30, edgecolor="k")
    ax.set_title(title)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (name, model) in zip(axes.ravel(), moon_models.items()):
    plot_cls_boundary(ax, model, X_moon, y_moon, name)
plt.tight_layout()
plt.show()

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=200,
        learning_rate=0.8,
        random_state=42,
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    fitted_models[name] = model
    results.append(
        {"模型": name, "Accuracy": accuracy_score(y_test, preds), "F1": f1_score(y_test, preds)}
    )

ensemble_df = pd.DataFrame(results).sort_values("F1", ascending=False)
ensemble_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=ensemble_df, x="模型", y="Accuracy", ax=axes[0], palette="Blues")
axes[0].set_title("集成模型 Accuracy 对比")

sns.barplot(data=ensemble_df, x="模型", y="F1", ax=axes[1], palette="Greens")
axes[1].set_title("集成模型 F1 对比")

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = ensemble_df.iloc[0]["模型"]
best_model = fitted_models[best_model_name]

importance = pd.Series(best_model.feature_importances_, index=X.columns)
top_importance = importance.sort_values(ascending=False).head(12).sort_values()

plt.figure(figsize=(10, 7))
plt.barh(top_importance.index, top_importance.values, color="slateblue")
plt.title(f"{best_model_name} 的 Top-12 特征重要性")
plt.xlabel("importance")
plt.show()